# COMP2501 · In-class Exercises 03 (interactive)

The official **Exercises 03**, rebuilt as a fill-in walkthrough. Topic: **web scraping with `rvest`**, then wrangling the scraped table with `dplyr` + `tidyr`.

**Packages needed:** `rvest`, `dplyr`, `tidyr` (all included in `tidyverse` except `rvest`).

```r
install.packages(c('rvest', 'dplyr', 'tidyr'))   # once, if needed
library(rvest)
library(dplyr)
library(tidyr)
```

> ⚠️ **Wikipedia changes.** The numbers in the answers below (239 athletes, 47 shooters, etc.) were correct *when the official answer key was written*. If you get a slightly different count, that's Wikipedia being edited, not you being wrong. The **method** is what matters.

> 🎓 These exercises combine everything from Lec 2-4: scraping → cleaning types → filtering → splitting columns → group-and-count.


---
## Exercise 1 — Scrape and clean

From this Wikipedia page, scrape the table **"List of athletes with at least six Olympic appearances"** and store it as `athletes`. Then:

- rename the variables
- convert `appearances`, `gold`, `silver`, `bronze` to **integers**
- **replace** the total-medal column with the sum of gold + silver + bronze
- print the dimensions, the first 6 rows, and the structure

**Source:** <https://en.wikipedia.org/wiki/List_of_athletes_with_the_most_appearances_at_Olympic_Games>

**Why replace `total`?** The original `Tot.` column contains a **footnote marker**, which makes R read it as *character* instead of a number. Recomputing it from the three medal columns sidesteps the problem entirely.


In [ ]:
# YOUR CODE
# 1. build the url (paste0 is handy for long urls)
# 2. page <- read_html(url)
# 3. table_node <- html_elements(page, 'table')[[4]]     # note: which table index?
# 4. raw_athletes <- html_table(table_node, fill = TRUE)
# 5. names(raw_athletes) <- c('appearances','athlete','representing',
#                            'birth_death','games','period','sport',
#                            'gold','silver','bronze','total')
# 6. convert types + recompute total
# 7. dim() / head() / str()

<details>
<summary><b>Reveal answer</b></summary>

```r
url <- paste0(
  'https://en.wikipedia.org/wiki/',
  'List_of_athletes_with_the_most_appearances_at_Olympic_Games'
)

page <- read_html(url)
table_node <- html_elements(page, 'table')[[4]]
raw_athletes <- html_table(table_node, fill = TRUE)

names(raw_athletes) <- c(
  'appearances', 'athlete', 'representing', 'birth_death', 'games',
  'period', 'sport', 'gold', 'silver', 'bronze', 'total'
)

athletes <- raw_athletes |>
  mutate_at(vars(appearances, gold, silver, bronze), as.integer) |>
  mutate(total = gold + silver + bronze)

dim(athletes)
head(athletes)
str(athletes)
```

**Expected:** 239 athletes × 11 variables, with `appearances`/`gold`/`silver`/`bronze`/`total` all integers.

**Two things worth noticing:**

1. `html_elements(page, 'table')[[4]]` — the page has several tables; index 4 is the one we want. *Always inspect* (`html_elements(page,'table')` then check each with `html_table()`) rather than guessing.
2. `mutate_at(vars(...), as.integer)` is a **scoped verb** — it applies `as.integer` to the listed columns. Modern dplyr prefers `across()`:
   ```r
   mutate(across(c(appearances, gold, silver, bronze), as.integer))
   ```
   Both work; `across()` is the current idiom. Use whichever your course expects.
</details>


---
## Exercise 2 — Who won the most of each medal?

Among these athletes, who has won the most **gold**, most **silver**, and most **bronze**? Keep **all ties** and report athlete, representing country/team, sport, and the medal count.


In [ ]:
# YOUR CODE
# For each medal type:
#   filter(medal == max(medal)) |>
#   select(athlete, representing, sport, <medal>)
#
# Careful: max() needs na.rm = TRUE if any medal column has NAs

<details>
<summary><b>Reveal answer</b></summary>

```r
most_gold <- athletes |>
  filter(gold == max(gold)) |>
  select(athlete, representing, sport, gold)

most_silver <- athletes |>
  filter(silver == max(silver)) |>
  select(athlete, representing, sport, silver)

most_bronze <- athletes |>
  filter(bronze == max(bronze)) |>
  select(athlete, representing, sport, bronze)

most_gold
most_silver
most_bronze
```

**Expected (at time of access):**
- Gold max = **8** — Isabell Werth, Birgit Fischer, Ole Einar Bjørndalen
- Silver max = **6** — Isabell Werth, Arianna Fontana
- Bronze max = **6** — Merlene Ottey, Harri Kirvesniemi

**The pattern to remember:** `filter(col == max(col))` is *the* idiom for "keep all ties at the top". It's much better than `top_n(1, col)`, which can silently drop ties (and multi-row `max` needs `na.rm` if NAs are present).
</details>


---
## Exercise 3 — Longest age span

Which athlete has the **longest age span** between their first and last Olympic appearances?

Required approach:
- use `separate()` to split the `Period (age of first/last)` column into `age_span` and `first_last_age`
- convert `age_span` to an integer
- keep all ties


In [ ]:
# YOUR CODE
# athletes |>
#   separate(period, into = c('age_span', 'first_last_age'),
#            sep = ' years ', remove = FALSE) |>
#   mutate(age_span = as.integer(age_span)) |>
#   filter(age_span == max(age_span)) |>
#   select(athlete, representing, games, period, age_span, first_last_age)
#
# Why sep = ' years ' and not sep = ' '?

<details>
<summary><b>Reveal answer</b></summary>

```r
longest_span <- athletes |>
  separate(period, into = c('age_span', 'first_last_age'),
           sep = ' years ', remove = FALSE) |>
  mutate(age_span = as.integer(age_span)) |>
  filter(age_span == max(age_span)) |>
  select(athlete, representing, games, period, age_span, first_last_age)

longest_span
```

**Expected:** max span = **40 years**, shared by Ian Millar, Durward Knowles, Paul Elvstrøm, Ivan Osiier, Magnus Konow.

**Key points:**

- **`sep = ' years '` (with spaces)**, not `sep = ' '`. The column looks like `'40 years (18/58)'`. Splitting on a bare space would also cut inside the parenthetical, and `into = c('age_span','first_last_age')` only allows 2 pieces — you'd get a "Additional pieces discarded" warning.
- **`remove = FALSE`** keeps the original `period` column so you can still see the raw value.
- `age_span` comes out as **character** (e.g. `'40'`), so `as.integer()` is required before `max()` works numerically. Without it, `max('40')` compares strings.
</details>


---
## Exercise 4 — Single-sport vs multi-sport athletes

Some athletes competed in one sport; others in several. Split them into `single_sport_athletes` and `multiple_sport_athletes`, and report the count in each.

**Hint:** the `sport` column uses `" and "` to join multiple sports (e.g. `'Bobsleigh and Luge'`).


In [ ]:
# YOUR CODE
# Use separate() with fill = 'right' and extra = 'merge' so that
# athletes with ONE sport get NA in the second column, and
# athletes with THREE sports don't error out.
#
# athletes_by_sport <- athletes |>
#   separate(sport, into = c('sport_before_and', 'sport_after_and'),
#            sep = ' and ', remove = FALSE, fill = 'right', extra = 'merge')
#
# single_sport_athletes  <- filter(is.na(sport_after_and))
# multiple_sport_athletes <- filter(!is.na(sport_after_and))

<details>
<summary><b>Reveal answer</b></summary>

```r
athletes_by_sport <- athletes |>
  separate(
    sport,
    into = c('sport_before_and', 'sport_after_and'),
    sep = ' and ',
    remove = FALSE,
    fill = 'right',
    extra = 'merge'
  )

single_sport_athletes <- athletes_by_sport |>
  filter(is.na(sport_after_and))

multiple_sport_athletes <- athletes_by_sport |>
  filter(!is.na(sport_after_and))

nrow(single_sport_athletes)     # 222
nrow(multiple_sport_athletes)   # 17
```

**Why the two arguments matter:**

- **`fill = 'right'`** — when an athlete has only ONE sport, there's nothing to put in the second column. Without `fill`, `separate` would either error or drop the row. `fill = 'right'` pads the missing right-hand piece with `NA`.
- **`extra = 'merge'`** — one athlete (Jaqueline Mourão) competed in **three** sports. Splitting into only 2 columns would discard the third. `extra = 'merge'` keeps the overflow joined to the last column, so she still contains `' and '` and lands correctly in `multiple_sport_athletes`.

**This is the same `extra`/`fill` pair from Lec 4's `separate()` section** — and this exercise is exactly why the lecture taught it.
</details>


---
## Exercise 5 — Which sport has the most six-time Olympians?

Among `single_sport_athletes`, which sport has the largest number of athletes? Keep all ties.


In [ ]:
# YOUR CODE
# single_sport_athletes |>
#   group_by(sport_before_and) |>
#   summarise(n = n()) |>
#   filter(n == max(n))

<details>
<summary><b>Reveal answer</b></summary>

```r
sport_counts <- single_sport_athletes |>
  group_by(sport_before_and) |>
  summarise(n = n())

sport_counts |>
  filter(n == max(n))
```

**Expected:** **Shooting**, with **47** athletes.

**The pattern:** `group_by(...) |> summarise(n = n()) |> filter(n == max(n))` is the general recipe for "most common category, keeping ties". You'll use it constantly.
</details>


---
## Recap

| Skill | Function(s) |
|---|---|
| Fetch a page | `read_html(url)` |
| Find tables | `html_elements(page, 'table')` |
| Convert to data frame | `html_table(node)` |
| Name columns | `setNames()` / `names() <-` |
| Convert types | `mutate_at(vars(...), as.integer)` or `mutate(across(...))` |
| Top values with ties | `filter(col == max(col))` |
| Split a column | `separate(..., sep=, fill=, extra=)` |
| Split into two groups | `filter(is.na(x))` / `filter(!is.na(x))` |
| Count per group | `group_by() |> summarise(n = n())` |

**The through-line:** scrape → clean types → reshape → summarize. Every real data project does this, and Exercises 03 is a complete miniature of it.
